In [21]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv
import time

In [3]:
load_dotenv()

model = ChatOpenAI()

In [4]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [5]:
def generate_joke(state:JokeState):
    prompt = f"Generate a joke based on the topic - {state['topic']}"
    response = model.invoke(prompt).content
    return {'joke':response}

In [6]:
def generate_explanation(state:JokeState):
    prompt = f"Based on the following joke on the topic {state['topic']}, generate an explanation\n{state['joke']}"
    response = model.invoke(prompt).content
    return {'explanation':response}

In [7]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke',generate_joke)
graph.add_node('generate_explanation',generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke','generate_explanation')
graph.add_edge('generate_explanation',END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [13]:
config1 = {"configurable":{"thread_id":'1'}}
initial_state = {'topic':'pizza'}

final_state = workflow.invoke(initial_state,config=config1)

final_state

{'topic': 'pizza',
 'joke': 'Why did the pizza go to the therapist?\n\nBecause it was stuffed crust with anxiety!',
 'explanation': 'Explanation: The joke suggests that the pizza went to see a therapist because it was feeling anxious. This is a playful way of humorously anthropomorphizing the pizza and implying that it has emotions and feelings just like a person would. It plays on the idea of "stuffed crust," a type of pizza crust that is filled with cheese, by suggesting that the pizza itself is "stuffed" with anxiety or feeling overwhelmed. This joke creates a funny scenario where a pizza, which is typically seen as an inanimate object, seeks therapy for its emotional issues.'}

In [14]:
workflow.get_state(config=config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the therapist?\n\nBecause it was stuffed crust with anxiety!', 'explanation': 'Explanation: The joke suggests that the pizza went to see a therapist because it was feeling anxious. This is a playful way of humorously anthropomorphizing the pizza and implying that it has emotions and feelings just like a person would. It plays on the idea of "stuffed crust," a type of pizza crust that is filled with cheese, by suggesting that the pizza itself is "stuffed" with anxiety or feeling overwhelmed. This joke creates a funny scenario where a pizza, which is typically seen as an inanimate object, seeks therapy for its emotional issues.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0fcb95-05b7-6f81-800a-bd13dfafc0c2'}}, metadata={'source': 'loop', 'step': 10, 'parents': {}}, created_at='2026-01-29T02:22:09.580838+00:00', parent_config={'configurable': {'thread_id': '1', 'chec

In [15]:
list(workflow.get_state_history(config=config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the therapist?\n\nBecause it was stuffed crust with anxiety!', 'explanation': 'Explanation: The joke suggests that the pizza went to see a therapist because it was feeling anxious. This is a playful way of humorously anthropomorphizing the pizza and implying that it has emotions and feelings just like a person would. It plays on the idea of "stuffed crust," a type of pizza crust that is filled with cheese, by suggesting that the pizza itself is "stuffed" with anxiety or feeling overwhelmed. This joke creates a funny scenario where a pizza, which is typically seen as an inanimate object, seeks therapy for its emotional issues.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0fcb95-05b7-6f81-800a-bd13dfafc0c2'}}, metadata={'source': 'loop', 'step': 10, 'parents': {}}, created_at='2026-01-29T02:22:09.580838+00:00', parent_config={'configurable': {'thread_id': '1', 'che

In [16]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti break up with the macaroni?\n\nBecause it heard he was too cheesy!',
 'explanation': 'This joke plays on the idea of macaroni being "cheesy" not just in flavor, but also in the sense of being overly affectionate or insincere. Spaghetti is known for being simple and straightforward, so the joke implies that it decided to break up with macaroni because it couldn\'t handle his cheesy behavior. It uses wordplay to create a humorously absurd reason for the relationship ending.'}

In [17]:
workflow.get_state(config=config2)

StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti break up with the macaroni?\n\nBecause it heard he was too cheesy!', 'explanation': 'This joke plays on the idea of macaroni being "cheesy" not just in flavor, but also in the sense of being overly affectionate or insincere. Spaghetti is known for being simple and straightforward, so the joke implies that it decided to break up with macaroni because it couldn\'t handle his cheesy behavior. It uses wordplay to create a humorously absurd reason for the relationship ending.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0fcb98-493a-6644-8002-612e4ed134a9'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-01-29T02:23:37.190355+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0fcb98-3c89-6cc9-8001-b4f6f99be5bf'}}, tasks=(), interrupts=())

In [33]:
list(workflow.get_state_history(config=config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti break up with the macaroni?\n\nBecause it heard he was too cheesy!', 'explanation': 'This joke plays on the idea of macaroni being "cheesy" not just in flavor, but also in the sense of being overly affectionate or insincere. Spaghetti is known for being simple and straightforward, so the joke implies that it decided to break up with macaroni because it couldn\'t handle his cheesy behavior. It uses wordplay to create a humorously absurd reason for the relationship ending.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0fcb98-493a-6644-8002-612e4ed134a9'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-01-29T02:23:37.190355+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0fcb98-3c89-6cc9-8001-b4f6f99be5bf'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did t

Fault Tolerance

In [22]:
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str
    step3: str

In [23]:
def step_1(state:CrashState):
    print("Step 1 executed")
    return {"step1":"done", "input":state["input"]}

def step_2(state:CrashState):
    print("Step 2 Crashing")
    time.sleep(30)
    return {"step2":"done"}

def step_3(state:CrashState):
    print("Step 3 executed")
    return {"step3":"done", "input":state["input"]}

In [24]:
builder = StateGraph(CrashState)
builder.add_node('step_1',step_1)
builder.add_node('step_2',step_2)
builder.add_node('step_3',step_3)

builder.add_edge(START,'step_1')
builder.add_edge('step_1','step_2')
builder.add_edge('step_2','step_3')
builder.add_edge('step_3',END)

graph = builder.compile(checkpointer=checkpointer)

In [26]:
try:
    print("running Graph")
    graph.invoke({"input":"done"},config=config1)
except KeyboardInterrupt:
    print("Crashed")

running Graph
Step 1 executed
Step 2 Crashing
Crashed


In [28]:
graph.get_state(config=config1)

StateSnapshot(values={'input': 'done', 'step1': 'done'}, next=('step_2',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0fcbb1-48be-6457-800d-5ced04f91bb3'}}, metadata={'source': 'loop', 'step': 13, 'parents': {}}, created_at='2026-01-29T02:34:48.228155+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0fcbb1-48bb-6d40-800c-ebb9349f136a'}}, tasks=(PregelTask(id='0bdc7353-a235-b5b9-71ce-33dfc2bb12f7', name='step_2', path=('__pregel_pull', 'step_2'), error=None, interrupts=(), state=None, result=None),), interrupts=())

In [29]:
list(graph.get_state_history(config=config1))

[StateSnapshot(values={'input': 'done', 'step1': 'done'}, next=('step_2',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0fcbb1-48be-6457-800d-5ced04f91bb3'}}, metadata={'source': 'loop', 'step': 13, 'parents': {}}, created_at='2026-01-29T02:34:48.228155+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0fcbb1-48bb-6d40-800c-ebb9349f136a'}}, tasks=(PregelTask(id='0bdc7353-a235-b5b9-71ce-33dfc2bb12f7', name='step_2', path=('__pregel_pull', 'step_2'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'input': 'done'}, next=('step_1',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0fcbb1-48bb-6d40-800c-ebb9349f136a'}}, metadata={'source': 'loop', 'step': 12, 'parents': {}}, created_at='2026-01-29T02:34:48.227155+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0fcbb1-48b8

In [30]:
final_graph = graph.invoke(None, config=config1)
final_state

Step 2 Crashing
Step 3 executed


{'topic': 'pizza',
 'joke': 'Why did the pizza go to the therapist?\n\nBecause it was stuffed crust with anxiety!',
 'explanation': 'Explanation: The joke suggests that the pizza went to see a therapist because it was feeling anxious. This is a playful way of humorously anthropomorphizing the pizza and implying that it has emotions and feelings just like a person would. It plays on the idea of "stuffed crust," a type of pizza crust that is filled with cheese, by suggesting that the pizza itself is "stuffed" with anxiety or feeling overwhelmed. This joke creates a funny scenario where a pizza, which is typically seen as an inanimate object, seeks therapy for its emotional issues.'}

In [31]:
graph.get_state(config=config1)

StateSnapshot(values={'input': 'done', 'step1': 'done', 'step2': 'done', 'step3': 'done'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0fcbb8-e2fb-671b-800f-42378726e54f'}}, metadata={'source': 'loop', 'step': 15, 'parents': {}}, created_at='2026-01-29T02:38:12.306101+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0fcbb8-e2fb-671a-800e-9e11003fbcd4'}}, tasks=(), interrupts=())

In [32]:
list(graph.get_state_history(config=config1))

[StateSnapshot(values={'input': 'done', 'step1': 'done', 'step2': 'done', 'step3': 'done'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0fcbb8-e2fb-671b-800f-42378726e54f'}}, metadata={'source': 'loop', 'step': 15, 'parents': {}}, created_at='2026-01-29T02:38:12.306101+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0fcbb8-e2fb-671a-800e-9e11003fbcd4'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'input': 'done', 'step1': 'done', 'step2': 'done'}, next=('step_3',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0fcbb8-e2fb-671a-800e-9e11003fbcd4'}}, metadata={'source': 'loop', 'step': 14, 'parents': {}}, created_at='2026-01-29T02:38:12.306101+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0fcbb1-48be-6457-800d-5ced04f91bb3'}}, tasks=(PregelTask(id='ce4586de-0d00-f86f-796d-f59c927fefc7', name='s

Time Travel

In [34]:
workflow.get_state(config={"configurable":{"thread_id":"2","checkpoint_id":'1f0fcb98-32af-688d-8000-76940a96ef6b'}})

StateSnapshot(values={'topic': 'pasta'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_id': '1f0fcb98-32af-688d-8000-76940a96ef6b'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-01-29T02:23:34.826612+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0fcb98-32ac-6a1d-bfff-de202ba7bfd5'}}, tasks=(PregelTask(id='74756aa4-da26-2e0b-13ba-a3858e66bcb6', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the spaghetti break up with the macaroni?\n\nBecause it heard he was too cheesy!'}),), interrupts=())

In [36]:
workflow.invoke(None,
config={"configurable":{"thread_id":"2","checkpoint_id":'1f0fcb98-32af-688d-8000-76940a96ef6b'}})

{'topic': 'pasta',
 'joke': "Why did the pasta chef break up with his girlfriend? \n\nBecause she couldn't handle his al dente deadlines!",
 'explanation': 'This joke plays on the term "al dente," which is an Italian term used to describe pasta that is cooked to be firm to the bite. In this context, the pasta chef\'s girlfriend couldn\'t handle his "al dente deadlines," suggesting that she couldn\'t cope with the high-pressure and strict deadlines that come with being a pasta chef. The punchline is a play on words, using the term "al dente" in a humorous way to explain why the pasta chef broke up with his girlfriend.'}

In [37]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': "Why did the pasta chef break up with his girlfriend? \n\nBecause she couldn't handle his al dente deadlines!", 'explanation': 'This joke plays on the term "al dente," which is an Italian term used to describe pasta that is cooked to be firm to the bite. In this context, the pasta chef\'s girlfriend couldn\'t handle his "al dente deadlines," suggesting that she couldn\'t cope with the high-pressure and strict deadlines that come with being a pasta chef. The punchline is a play on words, using the term "al dente" in a humorous way to explain why the pasta chef broke up with his girlfriend.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0fcc45-2397-6445-8002-51af64c11043'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-01-29T03:40:57.177197+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0fcc45-198e-6ca8-8001-07e17

In [38]:
workflow.update_state(config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0fcb98-32af-688d-8000-76940a96ef6b'}},values={'topic':'samosa'})

{'configurable': {'thread_id': '2',
  'checkpoint_ns': '',
  'checkpoint_id': '1f0fcc4e-0806-68dc-8001-114c99ccb853'}}

In [39]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0fcc4e-0806-68dc-8001-114c99ccb853'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-01-29T03:44:55.878678+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0fcb98-32af-688d-8000-76940a96ef6b'}}, tasks=(PregelTask(id='189f444a-7f86-db08-e9a9-a4f93c96cb33', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pasta', 'joke': "Why did the pasta chef break up with his girlfriend? \n\nBecause she couldn't handle his al dente deadlines!", 'explanation': 'This joke plays on the term "al dente," which is an Italian term used to describe pasta that is cooked to be firm to the bite. In this context, the pasta chef\'s girlfriend couldn\'t handle his "al dente

In [40]:
workflow.invoke(None,config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0fcc4e-0806-68dc-8001-114c99ccb853'}})

{'topic': 'samosa',
 'joke': 'Why did the samosa go to the party? Because it wanted to be the life of the samosa-ration!',
 'explanation': 'This joke plays on the wordplay between "samosa" and "party" and the phrase "life of the party." In the joke, the samosa went to the party because it wanted to be the center of attention and bring enjoyment to the event, similar to how someone might want to be the life of the party. The term "samosa-ration" is a pun on the word "celebration," implying that the samosa wanted to bring festivity and joy to the gathering. Overall, the joke is meant to be a playful and humorous play on words involving the popular Indian snack, the samosa.'}

In [41]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the samosa go to the party? Because it wanted to be the life of the samosa-ration!', 'explanation': 'This joke plays on the wordplay between "samosa" and "party" and the phrase "life of the party." In the joke, the samosa went to the party because it wanted to be the center of attention and bring enjoyment to the event, similar to how someone might want to be the life of the party. The term "samosa-ration" is a pun on the word "celebration," implying that the samosa wanted to bring festivity and joy to the gathering. Overall, the joke is meant to be a playful and humorous play on words involving the popular Indian snack, the samosa.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f0fcc52-61b5-6b43-8003-9648ab245214'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-01-29T03:46:52.656928+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_n